<a href="https://colab.research.google.com/github/pushpakgaidhane294/Gen_AI_Lab/blob/main/Practical3_Gen_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


# ============================================================
# 1. TRAINING DATA
# ============================================================

text = """
i am going home
i am going to school
i am going to college
i want to go home
i want to eat food
i want to play football
i want to learn python
i like to play football
i like to eat pizza
i like to learn programming
i love to play cricket
i love to eat pizza
i love programming
i am learning python
i am learning machine learning
i am learning deep learning
machine learning is interesting
deep learning is very powerful
python is easy to learn
python is a programming language
this is a good day
this is a beautiful day
this is my home
this is my college
how are you
how are you doing
what are you doing
what is your name
where are you going
where are you from
can you help me
can you help me with python
please help me
please give me some information
thank you for your help
have a nice day
see you tomorrow
good morning everyone
good night everyone
"""


# ============================================================
# 2. TOKENIZATION
# ============================================================

tokenizer = Tokenizer(
    oov_token="<OOV>"
)

tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1

print("Vocabulary size:", total_words)


# ============================================================
# 3. CREATE INPUT SEQUENCES
# ============================================================

input_sequences = []

sentences = text.strip().split("\n")

for sentence in sentences:

    token_list = tokenizer.texts_to_sequences([sentence])[0]

    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i + 1]
        input_sequences.append(n_gram_sequence)


# Find maximum sequence length
max_sequence_len = max(
    len(sequence) for sequence in input_sequences
)

print("Maximum sequence length:", max_sequence_len)


# ============================================================
# 4. PAD SEQUENCES
# ============================================================

input_sequences = np.array(
    pad_sequences(
        input_sequences,
        maxlen=max_sequence_len,
        padding="pre"
    )
)


# X = input words
# y = next word

X = input_sequences[:, :-1]
y = input_sequences[:, -1]


# ============================================================
# 5. CREATE LSTM MODEL
# ============================================================

model = Sequential()

model.add(
    Embedding(
        input_dim=total_words,
        output_dim=64,
        input_length=max_sequence_len - 1
    )
)

model.add(
    LSTM(128)
)

model.add(
    Dense(
        total_words,
        activation="softmax"
    )
)


# ============================================================
# 6. COMPILE MODEL
# ============================================================

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)


# ============================================================
# 7. TRAIN MODEL
# ============================================================

print("\nTraining model...\n")

model.fit(
    X,
    y,
    epochs=150,
    verbose=1
)

print("\nTraining completed!")


# ============================================================
# 8. NEXT WORD PREDICTION FUNCTION
# ============================================================

def predict_next_words(seed_text, number_of_words=5):

    seed_text = seed_text.lower().strip()

    # Convert input text to numbers
    token_list = tokenizer.texts_to_sequences(
        [seed_text]
    )[0]

    if len(token_list) == 0:
        return []

    # Keep only the last words if input is too long
    token_list = token_list[
        -(max_sequence_len - 1):
    ]

    # Add padding
    token_list = pad_sequences(
        [token_list],
        maxlen=max_sequence_len - 1,
        padding="pre"
    )

    # Predict probabilities
    probabilities = model.predict(
        token_list,
        verbose=0
    )[0]

    # Get highest probability word indexes
    top_indices = np.argsort(
        probabilities
    )[-number_of_words:][::-1]

    predictions = []

    # Convert indexes back to words
    reverse_word_index = {
        index: word
        for word, index in tokenizer.word_index.items()
    }

    for index in top_indices:

        word = reverse_word_index.get(index)

        if word is not None:
            probability = probabilities[index]

            predictions.append(
                (
                    word,
                    round(float(probability) * 100, 2)
                )
            )

    return predictions


# ============================================================
# 9. USER INPUT
# ============================================================

print("\n====================================")
print("     NEXT WORD PREDICTION SYSTEM")
print("====================================")

print("\nType a sentence and press Enter.")
print("Type 'exit' to stop.\n")


while True:

    user_input = input("Enter text: ")

    if user_input.lower().strip() == "exit":
        print("Program ended.")
        break

    if user_input.strip() == "":
        print("Please enter some text.")
        continue

    predictions = predict_next_words(
        user_input,
        number_of_words=5
    )

    print("\nSuggested next words:")

    if len(predictions) == 0:
        print("No prediction available.")

    else:


        for i, (word, probability) in enumerate(
            predictions,
            start=1
        ):
            print(
                f"{i}. {word} ({probability}%)"
            )

    print()

Vocabulary size: 63
Maximum sequence length: 6

Training model...

Epoch 1/150


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.0602 - loss: 4.1404   
Epoch 2/150
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0902 - loss: 4.1198
Epoch 3/150
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.0902 - loss: 4.0926
Epoch 4/150
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.0902 - loss: 4.0410
Epoch 5/150
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0902 - loss: 3.9294
Epoch 6/150
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0902 - loss: 3.7065
Epoch 7/150
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0902 - loss: 3.6353
Epoch 8/150
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0902 - loss: 3.5908
Epoch 9/150
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0902 - loss: 3.5321
Epoch 10/150
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.0977 - loss: 3.5227
Epoch 11/150
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.1504 - loss: 3.4924
Epoch 12/150
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.2030 - loss: 3.449

KeyboardInterrupt: Interrupted by user

In [ ]:
# Evaluate the model on the training data
loss, accuracy = model.evaluate(X, y, verbose=0)
print(f"Training Accuracy: {accuracy*100:.2f}%")

Training Accuracy: 72.18%
